code for generating simulated data for the bronze layer tables

Initialization steps  

In [0]:
%pip install faker

In [0]:
%pip install faker faker-commerce

In [0]:
volume_path ="/Volumes/workspace/default/amazon_fullfilment/source/"

Generating source data

In [0]:
#Generate Customers and addresses data

from pyspark.sql import Row
from faker import Faker
import random

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Initialize Faker with Canadian English locale
fake = Faker('en_CA')

def generate_customer_bundle(num_customers=100):
    customers = []
    addresses = []
    
    for _ in range(num_customers):
        # 1. Create unique Customer
        cust_id = str(fake.uuid4())
        customers.append(Row(
            customer_id=cust_id,
            first_name=fake.first_name(),
            last_name=fake.last_name(),
            gender=random.choice(["Male", "Female"]),
            email=fake.email(),
            phone=fake.phone_number()
        ))
        
        # 2. Create 1 to 3 addresses per Customer
        num_addr = random.randint(1, 3)
        for _ in range(num_addr):
            addresses.append(Row(
                address_id=str(fake.uuid4()),
                customer_id=cust_id, # Linking key
                street=fake.street_address(),
                city=fake.city(),
                province=fake.province_abbr(),
                postal_code=fake.postalcode()
            ))
            
    return spark.createDataFrame(customers), spark.createDataFrame(addresses)

# Generate the dataframes
customers_df, addresses_df = generate_customer_bundle(500)



# Save to source folder
(customers_df.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/customers"))
(addresses_df
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/addresses"))





In [0]:
#Generate products data

from pyspark.sql import Row
from faker import Faker
import faker_commerce
import random

fake = Faker('en_CA')
fake.add_provider(faker_commerce.Provider)

# 1. Define the Semantic Map
# This ensures that 'Electronics' only gets electronic-sounding names
PRODUCT_MAP = {
    "Electronics": {
        "brands": ["TechFlow", "ElectroNova", "VoltPulse", "AmazonBasics"],
        "items": ["Wireless Mouse", "Noise-Cancelling Headphones", "4K Monitor", "USB-C Hub", "Smart Watch"]
    },
    "Home & Kitchen": {
        "brands": ["PureComfort", "ChefMaster", "HearthStone", "AmazonBasics"],
        "items": ["Air Fryer", "Memory Foam Pillow", "Cast Iron Skillet", "Electric Kettle", "Microfiber Towel"]
    },
    "Groceries": {
        "brands": ["FreshPick", "Nature's Own", "OrganicHarvest", "WholeFoods"],
        "items": ["Artisanal Bacon", "Organic Honey", "Maple Syrup", "Cold Brew Coffee", "Aged Cheddar"]
    },
    "Music": {
        "brands": ["AudioWave", "EchoBeats", "MelodyMaker"],
        "items": ["Vinyl Record Player", "Acoustic Guitar Strings", "Studio Microphone", "Drum Sticks"]
    }
}

def generate_logical_products(num_products=500):
    products = []
    categories = list(PRODUCT_MAP.keys())
    
    for _ in range(num_products):
        # 2. Pick the Category first
        category = random.choice(categories)
        
        # 3. Use the Category to pick matching brands and items
        brand = random.choice(PRODUCT_MAP[category]["brands"])
        item = random.choice(PRODUCT_MAP[category]["items"])
        
        product_name = f"{brand} {item}"
        
        if random.random() < 0.7:
                 price = round(random.uniform(5.00, 50.00), 2)
        else:
                 price = round(random.uniform(50.01, 800.00), 2)
                 
        products.append(Row(
            product_id=str(fake.uuid4()),
            sku=f"{category[:3].upper()}-{random.randint(100, 999)}",
            product_name=product_name,
            category=category,
            #price=float(fake.ecommerce_price()),
            weight_kg=round(random.uniform(0.1, 10.0), 2)
        ))
    
    return spark.createDataFrame(products)

# Test it
products_df = generate_logical_products()
(products_df.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/products"))



In [0]:
#Generate robots data

from pyspark.sql import Row
from faker import Faker
import random
from datetime import datetime

def generate_robot_batch(num_records=10):
    fake = Faker()
    # Mock IDs for our 10 robots
    robot_ids = [f"ROB_{i:03d}" for i in range(1, 11)]
    
    data = []
    for _ in range(num_records):
        now = datetime.now()
        record = Row(
            event_id=str(fake.uuid4()),
            robot_id=random.choice(robot_ids),
            battery_level=random.randint(15, 100),
            status=random.choice(["Moving", "Stowing", "Picking", "Charging"]),
            location_zone=random.choice(["Zone-A", "Zone-B", "Zone-C"]),
            Processed_date=now.date(),
            timestamp=datetime.now()
        )
        data.append(record)
    
    return spark.createDataFrame(data)

# Run this every time the notebook starts
new_events_df = generate_robot_batch(1000)

(new_events_df.write
    .format("csv")
    .mode("append")
    .option("header",True)
    .option("delimiter",",")
    .save(f"{volume_path}/robots"))
                 

In [0]:
# Generate Orders data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.customers"):
    customer_id = [row['Customer_ID'] for row in spark.table("workspace.amazon_fullfilment_bronze.customers").select("Customer_ID").collect()]

    def generate_orders_data(num_orders=50):
        orders = []
        from datetime import datetime, timedelta
        today = datetime.now() 

        
        for _ in range(num_orders):
            order_id = str(uuid.uuid4())
            
            # 1. Logic: Go back between 0 and 14 days
            days_ago = random.randint(0, 14)
            order_date = today - timedelta(days=days_ago)
            
            # 2. Logic: Status should ideally match the age
            # New orders (0-1 days old) are likely Pending/Picking
            # Older orders (5+ days) are likely Packed/Shipped
            status = "Initial"
           # if days_ago < 2:
            #    status = random.choice(["Pending", "Picking"])
            #elif days_ago < 5:
            #    status = random.choice(["Picking", "Packed"])
            #else:
            #    status = random.choice(["Packed", "Shipped"])

            orders.append(Row(
                Order_ID=order_id,
                Customer_ID=random.choice(customer_id), 
                OrderDate=order_date.date(), # Returns a real Date object, not a string
                Status=status
            ))
            
        return spark.createDataFrame(orders)

    # Generate and view
    orders_df = generate_orders_data()

    # save to the source volume
    (orders_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/orders"))
else:
    print("Warning: customers table not found. Skipping.")


In [0]:
# Generate Orders_Item data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.products") and spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.orders"):
    order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").filter("status='Initial'").collect()]
    product_id = [row['product_id'] for row in spark.table("workspace.amazon_fullfilment_bronze.products").select("product_id").collect()]

    def generate_order_items_data(num_order_items=200):
        order_items = []
        for _ in range(num_order_items):
            order_item_id = str(uuid.uuid4())
            #order_id = 
            #product_id = 
            quantity = random.randint(1, 10)
            order_items.append(Row(
                OrderItemID=order_item_id,
                OrderID=random.choice(order_id),
                ProductID=random.choice(product_id),
                Quantity=quantity
            ))
            
        return spark.createDataFrame(order_items)

    # Generate and view
    order_items_df = generate_order_items_data()

    # save to the source volume

    (order_items_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/order_items"))
else:
    print("Warning: Products and Orders table not found. Skipping.")


 

In [0]:
#Generate Shelves and Inventory data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.products") and spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.robots"):
    # 1. Get existing IDs to ensure referential integrity
    product_ids = [row['ProductID'] for row in spark.table("workspace.amazon_fullfilment_bronze.order_items").select("ProductID").collect()]
    robot_ids = [row['RobotID'] for row in spark.table("workspace.amazon_fullfilment_bronze.robots").select("RobotID").filter("Status='Picking'").collect()]

    def generate_warehouse_data(num_shelves=500):
        shelves = []
        inventory = []
        
        for _ in range(num_shelves):
            shelf_id = str(uuid.uuid4())
            
            # Create the Shelf
            # We assign a RobotID to some shelves to simulate robots currently carrying them
            assigned_robot = random.choice(robot_ids) if random.random() > 0.5 else None
            
            shelves.append(Row(
                ShelfID=shelf_id,
                CurrentRobotID=assigned_robot,
                MaxWeightCapacity=float(random.uniform(500.0, 1500.0))
            ))
            
            # Create Shelf Inventory (Each shelf holds 3-10 random unique products)
            selected_products = random.sample(product_ids, random.randint(3, 10))
            for p_id in selected_products:
                inventory.append(Row(
                    ShelfID=shelf_id,
                    ProductID=p_id,
                    Quantity=random.randint(1, 50)
                ))
                
        return spark.createDataFrame(shelves), spark.createDataFrame(inventory)

    # Generate and view
    shelves_df, inventory_df = generate_warehouse_data()

    # save to the source volume

    (shelves_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/shelves"))


    (inventory_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/inventory"))

    spark.sql("UPDATE workspace.amazon_fullfilment_bronze.orders SET Status ='Inventory' WHERE order_id in f{order_id}")
else:
    print("Warning: Products and Orders table not found. Skipping.")


In [0]:
# Generate Bin data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.orders"):
    order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").filter("Status='Transitioning'").collect()]

    def generate_bin_data(num_bins=50):
        bins = []
        for _ in range(num_bins):
            bin_id = str(uuid.uuid4())
            bin_location = random.choice(["warehouse", "shipping"])
            bin_capacity = random.randint(100, 500)
            bin_weight = random.randint(10, 100)
            bins.append(Row(
                BinID=bin_id,
                CurrentOrderID=random.choice(order_id),
                BinLocation=bin_location,
                BinCapacity=bin_capacity,
                BinWeight=bin_weight
            ))
            
        return spark.createDataFrame(bins)

    # Generate and view
    bin_df = generate_bin_data()

    # save to the source volume
    (bin_df.write
    .format("csv")
    .mode("append")
    .option("header",True)
    .option("delimiter",",")
    .save(f"{volume_path}/bin"))

    spark.sql("UPDATE workspace.amazon_fullfilment_bronze.orders SET Status ='Picking' WHERE order_id in f{order_id}")
else:
    print("Warning: Orders table not found. Skipping.")


In [0]:
# Generate Shipment data

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

if spark.catalog.tableExists("workspace.amazon_fullfilment_bronze.orders"):
    order_id = [row['orderid'] for row in spark.table("workspace.amazon_fullfilment_bronze.orders").select("orderid").collect()]

    def generate_shipment_data(num_shipments=50):
        shipments = []
        for _ in range(num_shipments):
            shipment_id = str(uuid.uuid4())
            tracking_number = random.randint(10000000, 9999999999)
            Shipping_label_url = f"https://www.example.com/labels/{shipment_id}"
            vehicle_id = str(uuid.uuid4())
            shipments.append(Row(
                ShipmentID=shipment_id,
                OrderID=random.choice(order_id),
                TrackingNumber=tracking_number,
                ShippingLabelURL=Shipping_label_url,
                VehicleID=vehicle_id
                ))
            
        return spark.createDataFrame(shipments)

    # Generate and view
    shipment_df = generate_shipment_data()

    # save to the source volume
    (shipment_df.write
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .mode("append")
    .save(f"{volume_path}/shipment"))
else:
    print("Warning: Orders table not found. Skipping.")
